# SD.NEXT Colab Bridge

Roda o SD.NEXT no Google Colab usando a GPU da nuvem, mas mantendo os modelos na sua maquina local.

## Como funciona
1. **Sua maquina local** roda um servidor HTTP + cloudflared tunnel que expoe seus modelos
2. **O Colab** clona o SD.NEXT, baixa modelos sob demanda do seu PC, e roda com a GPU do Colab
3. **WebUI** fica acessivel via outro tunnel cloudflared
4. **Google Drive** persiste tudo (SD.NEXT, venv, modelos, outputs) entre sessoes

## Pre-requisitos
- Rode `start_server.bat` na sua maquina local ANTES de executar este notebook
- Copie a URL do tunnel (*.trycloudflare.com) que aparece no terminal
- Cole no campo `LOCAL_SERVER_URL` abaixo

## Primeira sessao vs Proximas
- **Primeira vez**: Clone + instalacao completa (~15-20 min). Tudo fica salvo no Drive.
- **Proximas sessoes**: Restauracao instantanea do Drive. So precisa colar a nova URL do tunnel.

---
## ⚡ Quick Resume (Sessões Seguintes)

**Use esta célula SOMENTE se já rodou o setup completo (células 1-6) pelo menos uma vez.**

Ela restaura o ambiente inteiro do Google Drive em ~1 minuto:
- Monta o Drive e recria os symlinks
- Reinstala o cloudflared (binário do sistema, não persiste entre sessões)
- Modelos, venv e SD.NEXT já estão no Drive

**Depois de rodar esta célula, pule direto para a Célula 6 (Start SD.NEXT).**

In [ ]:
# ============================================================
# QUICK RESUME - Restaurar ambiente do Google Drive (~1 min)
# ============================================================
# Rode esta celula e pule direto para a Celula 6 (Start SD.NEXT)
# NAO rode se eh sua primeira vez - use o fluxo completo (celulas 1-6)
# ============================================================
import subprocess, os, shutil, sys, json

DRIVE_PATH = '/content/drive/MyDrive/SD_Data'
DRIVE_SDNEXT = f'{DRIVE_PATH}/sdnext'
LOCAL_SDNEXT = '/content/sdnext'

# --- [1/5] Montar Google Drive ---
print('--- [1/5] MONTANDO GOOGLE DRIVE ---')
from google.colab import drive
drive.mount('/content/drive')

# --- Verificar se cache existe ---
if not os.path.exists(f'{DRIVE_SDNEXT}/launch.py'):
    raise RuntimeError(
        'Cache NAO encontrado no Drive!\n'
        'Rode o setup completo (celulas 1-6) pelo menos uma vez primeiro.'
    )

# --- [2/5] Restaurar symlinks ---
print('\n--- [2/5] RESTAURANDO SD.NEXT DO DRIVE ---')
if os.path.exists(LOCAL_SDNEXT):
    if os.path.islink(LOCAL_SDNEXT):
        os.unlink(LOCAL_SDNEXT)
    else:
        shutil.rmtree(LOCAL_SDNEXT)
os.symlink(DRIVE_SDNEXT, LOCAL_SDNEXT)

# Verificar o que esta no cache
has_venv = os.path.exists(f'{LOCAL_SDNEXT}/venv')
models_dir = f'{LOCAL_SDNEXT}/models/Stable-diffusion'
model_files = [f for f in os.listdir(models_dir)
               if f.endswith(('.safetensors', '.ckpt', '.pt'))] if os.path.isdir(models_dir) else []

print(f'  Venv:    {"OK" if has_venv else "Nao encontrado (sera criado no launch)"}')
print(f'  Modelos: {len(model_files)} checkpoint(s) encontrado(s)')
for m in model_files:
    size = os.path.getsize(f'{models_dir}/{m}') / (1024**3)
    print(f'           - {m} ({size:.1f} GB)')

# Listar todas as categorias com modelos
model_exts = {'.safetensors', '.ckpt', '.pt', '.pth', '.bin'}
base = f'{LOCAL_SDNEXT}/models'
total_models = 0
for cat in sorted(os.listdir(base)):
    cat_path = f'{base}/{cat}'
    if not os.path.isdir(cat_path):
        continue
    files = [f for f in os.listdir(cat_path) if os.path.splitext(f)[1].lower() in model_exts]
    if files and cat != 'Stable-diffusion':
        total_models += len(files)
        print(f'  {cat}: {len(files)} arquivo(s)')
total_models += len(model_files)

# --- [3/5] Restaurar dependencias pip do checkpoint ---
print('\n--- [3/5] RESTAURANDO DEPENDENCIAS PIP ---')
req_file = f'{DRIVE_PATH}/requirements_colab.txt'
if os.path.exists(req_file):
    print('  Encontrado requirements_colab.txt do checkpoint')
    print('  Instalando pacotes (isso leva ~1-2 min)...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', '-r', req_file],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  OK - dependencias restauradas')
    else:
        # Tentar instalar ignorando erros individuais
        failed = []
        with open(req_file) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or line.startswith('-'):
                    continue
                r = subprocess.run(
                    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', line],
                    capture_output=True
                )
                if r.returncode != 0:
                    failed.append(line.split('==')[0])
        if failed:
            print(f'  Aviso: {len(failed)} pacote(s) falharam (nao-criticos): {", ".join(failed[:5])}')
        else:
            print('  OK - dependencias restauradas')
else:
    print('  requirements_colab.txt nao encontrado no Drive')
    print('  (Rode o Checkpoint pelo menos uma vez para salvar as deps)')

# --- Mostrar ultimo checkpoint ---
manifest_path = f'{DRIVE_PATH}/checkpoint_manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f'\n  Ultimo checkpoint: {manifest.get("timestamp", "?")[:19]}')

# --- [4/5] Instalar cloudflared ---
print('\n--- [4/5] CLOUDFLARED ---')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('  Instalando cloudflared...')
    subprocess.run(['wget', '-q',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('  OK')
else:
    print('  Ja instalado')

# --- [5/5] GPU + Disco ---
print('\n--- [5/5] STATUS ---')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'  GPU:   {gpu.stdout.strip()}')

total_d, used_d, free_d = shutil.disk_usage('/')
print(f'  Disco: {free_d//(1024**3)} GB livres')
total_g, used_g, free_g = shutil.disk_usage('/content/drive')
print(f'  Drive: {used_g//(1024**3)} GB usados | {free_g//(1024**3)} GB livres')

# --- Resumo ---
print('\n' + '='*60)
print(f'  AMBIENTE RESTAURADO! ({total_models} modelos no cache)')
print('  Pule direto para a CELULA 6 (Start SD.NEXT)')
print('='*60)

--- [1/5] MONTANDO GOOGLE DRIVE ---
Mounted at /content/drive


RuntimeError: Cache NAO encontrado no Drive!
Rode o setup completo (celulas 1-6) pelo menos uma vez primeiro.

---
## 1. Setup: GPU Check + Dependencias

In [ ]:
# ============================================================
# Cell 1: Setup Completo (Hardware + Dependências Inteligentes)
# ============================================================
import subprocess
import os
import shutil
import sys
import pkg_resources

def get_installed_version(package_name):
    """Retorna a versão instalada de um pacote ou None se não encontrado."""
    try:
        return pkg_resources.get_distribution(package_name).version
    except pkg_resources.DistributionNotFound:
        return None

def setup_environment():
    print("--- [1/3] VERIFICANDO HARDWARE E INFRA ---")

    # --- GPU Check ---
    gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print(f"✓ GPU: {gpu_info.stdout.strip()}")
    else:
        print("! AVISO: Nenhuma GPU detectada. Verifique 'Ambiente de Execução'.")

    # --- Disk space ---
    total, used, free = shutil.disk_usage('/')
    print(f"✓ Disco: {free//(1024**3)} GB livres de {total//(1024**3)} GB")

    # --- Cloudflared ---
    if not os.path.exists('/usr/local/bin/cloudflared'):
        print("-> Instalando cloudflared...")
        subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'])
        subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
    else:
        print("✓ cloudflared já está pronto.")

    print("\n--- [2/3] VERIFICANDO VERSÕES DAS BIBLIOTECAS ---")

    # Lista de dependências críticas (Nome, Versão Alvo)
    packages_to_fix = [
        ("gradio", "3.43.2"),
        ("fastapi", "0.124.4"),
        ("rich", "14.1.0"),
        ("numpy", "2.1.2"),
        ("pandas", "2.3.1"),
        ("numba", "0.61.2"),
        ("protobuf", "4.25.3"),
        ("urllib3", "1.26.19"),
        ("Pillow", "10.4.0"),
        ("timm", "1.0.16"),
        ("pyparsing", "3.2.3"),
        ("typing-extensions", "4.14.1"),
        ("scipy", "1.14.1"),
        ("transformers", "4.57.5"),
        ("huggingface_hub", "0.36.0"),
        ("einops", "0.8.1"),
        ("tqdm", "4.67.1")
    ]

    needed_install = []

    for name, target_version in packages_to_fix:
        current_version = get_installed_version(name)

        if current_version == target_version:
            print(f"  [OK] {name}=={current_version}")
        else:
            status = f"Atualizar: {current_version} -> {target_version}" if current_version else f"Instalar: {target_version}"
            print(f"  [!] {name}: {status}")
            needed_install.append((name, target_version))

    if needed_install:
        print(f"\nSincronizando {len(needed_install)} pacotes...")
        for name, version in needed_install:
            # Desinstala apenas se a versão estiver errada (evita erro se não existir)
            if get_installed_version(name):
                subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", name], capture_output=True)

            # Instala a versão correta
            print(f" -> Instalando {name}=={version}...", end=" ")
            try:
                subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--quiet", f"{name}=={version}"], check=True)
                print("✓")
            except:
                print(f"Falha na versão fixa. Tentando {name} mais recente...")
                subprocess.run([sys.executable, "-m", "pip", "install", name], check=True)

        print("\n--- [3/3] APLICANDO PATCHES FINAIS ---")
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", "torchvision", "diffusers"], capture_output=True)
        print("✓ Patch de compatibilidade aplicado.")

        print("\n" + "="*70)
        print("REINICIE A SESSÃO: Vá em Ambiente de Execução -> Reiniciar sessão")
        print("Isso é necessário para aplicar as novas versões na memória.")
        print("="*70)
    else:
        print("\n--- [3/3] AMBIENTE JÁ ESTÁ OTIMIZADO ---")
        print("✓ Tudo pronto para iniciar o SD.NEXT.")

if __name__ == "__main__":
    setup_environment()

/tmp/ipython-input-4203749736.py:8: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


--- [1/3] VERIFICANDO HARDWARE E INFRA ---
✓ GPU: Tesla T4, 15360 MiB, 580.82.07
✓ Disco: 70 GB livres de 112 GB
-> Instalando cloudflared...

--- [2/3] VERIFICANDO VERSÕES DAS BIBLIOTECAS ---
  [!] gradio: Atualizar: 5.50.0 -> 3.43.2
  [!] fastapi: Atualizar: 0.129.0 -> 0.124.4
  [!] rich: Atualizar: 13.9.4 -> 14.1.0
  [!] numpy: Atualizar: 2.0.2 -> 2.1.2
  [!] pandas: Atualizar: 2.2.2 -> 2.3.1
  [!] numba: Atualizar: 0.60.0 -> 0.61.2
  [!] protobuf: Atualizar: 5.29.6 -> 4.25.3
  [!] urllib3: Atualizar: 2.5.0 -> 1.26.19
  [!] Pillow: Atualizar: 11.3.0 -> 10.4.0
  [!] timm: Atualizar: 1.0.24 -> 1.0.16
  [!] pyparsing: Atualizar: 3.3.2 -> 3.2.3
  [!] typing-extensions: Atualizar: 4.15.0 -> 4.14.1
  [!] scipy: Atualizar: 1.16.3 -> 1.14.1
  [!] transformers: Atualizar: 5.0.0 -> 4.57.5
  [!] huggingface_hub: Atualizar: 1.4.1 -> 0.36.0
  [!] einops: Atualizar: 0.8.2 -> 0.8.1
  [!] tqdm: Atualizar: 4.67.3 -> 4.67.1

Sincronizando 17 pacotes...
 -> Instalando gradio==3.43.2... ✓
 -> Instaland

---
## 2. Google Drive + SD.NEXT (com cache persistente)

Na primeira execucao, clona o SD.NEXT e salva no Drive.
Nas proximas sessoes, restaura tudo instantaneamente (venv, modelos, configs).

In [ ]:
# ============================================================
# Cell 2: Google Drive + SD.NEXT (Clone ou Restaurar do Cache)
# ============================================================
from google.colab import drive
import os, shutil

# --- Montar Google Drive ---
print('Montando Google Drive...')
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/SD_Data'
DRIVE_SDNEXT = f'{DRIVE_PATH}/sdnext'
LOCAL_SDNEXT = '/content/sdnext'

os.makedirs(DRIVE_PATH, exist_ok=True)

# --- Verificar se SD.NEXT ja esta no Drive (sessao anterior) ---
if os.path.exists(f'{DRIVE_SDNEXT}/launch.py'):
    print('\n--- CACHE ENCONTRADO NO DRIVE! ---')

    # Criar symlink do Drive para o local
    if os.path.exists(LOCAL_SDNEXT):
        if os.path.islink(LOCAL_SDNEXT):
            os.unlink(LOCAL_SDNEXT)
        else:
            shutil.rmtree(LOCAL_SDNEXT)
    os.symlink(DRIVE_SDNEXT, LOCAL_SDNEXT)

    # Verificar o que ja esta cacheado
    has_venv = os.path.exists(f'{LOCAL_SDNEXT}/venv')
    has_models = any(os.listdir(f'{LOCAL_SDNEXT}/models/Stable-diffusion')) if os.path.isdir(f'{LOCAL_SDNEXT}/models/Stable-diffusion') else False

    print(f'  Venv (bibliotecas): {"Encontrado - instalacao sera pulada!" if has_venv else "Nao encontrado - sera criado no primeiro launch"}')
    print(f'  Modelos:            {"Encontrados no cache!" if has_models else "Precisam ser baixados via tunnel"}')

    # Tentar atualizar o repo (silencioso, nao falha se sem internet)
    print('  Verificando atualizacoes...', end=' ')
    ret = os.system(f'cd {LOCAL_SDNEXT} && git pull --ff-only 2>/dev/null')
    print('Atualizado!' if ret == 0 else 'Usando versao do cache.')

    print('\n  Setup INSTANTANEO concluido!')

else:
    print('\n--- PRIMEIRA EXECUCAO ---')
    print('Clonando SD.NEXT (sera salvo no Drive para proximas sessoes)...\n')

    # Clonar direto no Drive (tudo ja fica persistido)
    if os.path.exists(LOCAL_SDNEXT):
        shutil.rmtree(LOCAL_SDNEXT)

    !git clone --depth 1 https://github.com/vladmandic/sdnext.git {DRIVE_SDNEXT}

    # Symlink para o caminho esperado
    os.symlink(DRIVE_SDNEXT, LOCAL_SDNEXT)

    print('\nSD.NEXT clonado e salvo no Drive!')
    print('Na proxima sessao, sera restaurado instantaneamente.')
    print('(O primeiro launch vai instalar as bibliotecas no venv - isso tambem fica salvo)')

# --- Garantir que diretorios de modelos existam ---
MODEL_CATEGORIES = [
    'Stable-diffusion', 'Lora', 'VAE', 'embeddings',
    'controlnet', 'hypernetworks', 'ESRGAN', 'upscale_models',
    'text_encoders', 'UNET', 'clip', 'clip_vision',
]
for cat in MODEL_CATEGORIES:
    os.makedirs(f'{LOCAL_SDNEXT}/models/{cat}', exist_ok=True)

# --- Criar pasta de outputs ---
os.makedirs(f'{LOCAL_SDNEXT}/outputs', exist_ok=True)

# --- Status do Drive ---
total, used, free = shutil.disk_usage('/content/drive')
print(f'\nGoogle Drive: {used//(1024**3)} GB usados | {free//(1024**3)} GB livres')

Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- PRIMEIRA EXECUCAO ---
Clonando SD.NEXT (sera salvo no Drive para proximas sessoes)...

Cloning into '/content/drive/MyDrive/SD_Data/sdnext'...
remote: Enumerating objects: 1670, done.
remote: Counting objects: 100% (1670/1670), done.
remote: Compressing objects: 100% (1467/1467), done.
remote: Total 1670 (delta 202), reused 882 (delta 155), pack-reused 0 (from 0)
Receiving objects: 100% (1670/1670), 19.34 MiB | 5.12 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Updating files: 100% (1509/1509), done.

SD.NEXT clonado e salvo no Drive!
Na proxima sessao, sera restaurado instantaneamente.
(O primeiro launch vai instalar as bibliotecas no venv - isso tambem fica salvo)

Google Drive: 48 GB usados | 64 GB livres


---
## 3. Conectar ao Servidor Local de Modelos

**IMPORTANTE:** Rode `start_server.bat` na sua maquina local e copie a URL do tunnel aqui.

In [ ]:
# ============================================================
# Cell 3: Conectar ao servidor local e listar modelos
# ============================================================
import requests, json, time, os, shutil
from pathlib import Path

#@markdown ### Cole a URL do cloudflared tunnel da sua maquina local:
LOCAL_SERVER_URL = 'https://journalist-theory-button-trail.trycloudflare.com   ' #@param {type:"string"}

# Limpar URL
LOCAL_SERVER_URL = LOCAL_SERVER_URL.strip().rstrip('/')
if not LOCAL_SERVER_URL:
    raise ValueError('Cole a URL do tunnel no campo LOCAL_SERVER_URL acima!')

# Testar conexao
print(f'Conectando a {LOCAL_SERVER_URL}...')
try:
    r = requests.get(f'{LOCAL_SERVER_URL}/api/health', timeout=15)
    r.raise_for_status()
    info = r.json()
    print(f'Conectado! Servidor local OK.')
    print(f'Models dir: {info.get("models_dir", "N/A")}')
except Exception as e:
    raise ConnectionError(
        f'Falha ao conectar: {e}\n'
        f'Verifique se start_server.bat esta rodando na sua maquina.'
    )

# Listar modelos disponiveis
print('\n' + '='*60)
print('  MODELOS DISPONIVEIS NA SUA MAQUINA LOCAL')
print('='*60)

r = requests.get(f'{LOCAL_SERVER_URL}/api/models', timeout=30)
all_models = r.json()

for category, info in sorted(all_models.items()):
    model_files = [f for f in info['files']
                   if f['name'].endswith(('.safetensors', '.ckpt', '.pt', '.pth', '.bin'))]
    if not model_files:
        continue
    total_mb = sum(f['size_mb'] for f in model_files)
    print(f'\n[{category}] ({len(model_files)} modelos, {total_mb:.0f} MB total)')
    for i, f in enumerate(model_files, 1):
        size_str = f'{f["size_mb"]:.0f} MB' if f['size_mb'] < 1024 else f'{f["size_mb"]/1024:.1f} GB'
        print(f'  {i:3d}. {f["name"]}  ({size_str})')

print('\n' + '='*60)
print('Use a proxima celula para baixar os modelos desejados.')

Conectando a https://journalist-theory-button-trail.trycloudflare.com...
Conectado! Servidor local OK.
Models dir: G:\Local_AI\Img_Generators\Models

  MODELOS DISPONIVEIS NA SUA MAQUINA LOCAL

[Lora] (52 modelos, 12360 MB total)
    1. 69yottea_illu_v2.safetensors  (218 MB)
    2. 748cm_c_illu.safetensors  (218 MB)
    3. 748cmHassaku.safetensors  (244 MB)
    4. 748cmSDXL.safetensors  (243 MB)
    5. 96yottea-000123.safetensors  (268 MB)
    6. 96YOTTEA-WAI.safetensors  (218 MB)
    7. Anime_artistic_2.safetensors  (218 MB)
    8. Anime_in_real.safetensors  (218 MB)
    9. ass_worship_v0.3-illu_done.safetensors  (55 MB)
   10. Body Type Slider - Illustrious_alpha1.0_rank4_noxattn_last.safetensors  (8 MB)
   11. Breast Size Slider - Illustrious - V2_alpha1.0_rank4_noxattn_last.safetensors  (8 MB)
   12. BSS_OIL_PAINTING.safetensors  (218 MB)
   13. cfg_scale_boost.safetensors  (163 MB)
   14. chxrrygxg_v1-illustrious-ty_lee .safetensors  (435 MB)
   15. ck-shadow-circuit-IL-000012.saf

---
## 4. Model Manager - Baixar, Listar Cache e Limpar

In [ ]:
# ============================================================
# Cell 4: Model Manager class
# ============================================================
import requests, os, shutil, time, sys
from pathlib import Path

SDNEXT_DIR = '/content/sdnext'

class ModelManager:
    """Gerencia modelos entre maquina local e Colab."""

    def __init__(self, server_url, models_path=None):
        self.server_url = server_url.rstrip('/')
        self.models_path = models_path or f'{SDNEXT_DIR}/models'

    def download(self, category, filename, force=False):
        """Baixa um modelo do servidor local para o Colab."""
        dest = Path(self.models_path) / category / filename
        dest.parent.mkdir(parents=True, exist_ok=True)

        # Verificar se ja existe
        if dest.exists() and not force:
            print(f'Modelo ja existe: {dest.name} ({dest.stat().st_size/(1024**2):.0f} MB)')
            return str(dest)

        url = f'{self.server_url}/download/{category}/{filename}'

        # Verificar espaco em disco
        _, _, free = shutil.disk_usage('/')
        print(f'Espaco livre: {free//(1024**3)} GB')

        # Suporte a download resumivel
        headers = {}
        mode = 'wb'
        existing_size = 0
        temp_path = dest.with_suffix(dest.suffix + '.downloading')

        if temp_path.exists():
            existing_size = temp_path.stat().st_size
            headers['Range'] = f'bytes={existing_size}-'
            mode = 'ab'
            print(f'Retomando download de {existing_size/(1024**2):.0f} MB...')

        print(f'Baixando: {category}/{filename}')
        start = time.time()

        try:
            r = requests.get(url, headers=headers, stream=True, timeout=30)
            r.raise_for_status()

            if r.status_code == 206:  # Partial content (resumed)
                total = existing_size + int(r.headers.get('Content-Length', 0))
            else:
                total = int(r.headers.get('Content-Length', 0))
                existing_size = 0
                mode = 'wb'

            downloaded = existing_size
            chunk_size = 1024 * 1024  # 1MB

            with open(temp_path, mode) as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        elapsed = time.time() - start
                        speed = (downloaded - existing_size) / elapsed if elapsed > 0 else 0
                        speed_str = f'{speed/(1024**2):.1f} MB/s' if speed > 0 else '...'
                        pct = (downloaded / total * 100) if total > 0 else 0
                        remaining = (total - downloaded) / speed if speed > 0 else 0
                        eta = f'{remaining/60:.0f}min' if remaining > 60 else f'{remaining:.0f}s'
                        bar_len = 30
                        filled = int(bar_len * pct / 100)
                        bar = '█' * filled + '░' * (bar_len - filled)
                        sys.stdout.write(
                            f'\r  [{bar}] {pct:.1f}% | '
                            f'{downloaded/(1024**2):.0f}/{total/(1024**2):.0f} MB | '
                            f'{speed_str} | ETA: {eta}    '
                        )
                        sys.stdout.flush()

            # Rename temp to final
            if temp_path.exists():
                if dest.exists():
                    dest.unlink()
                temp_path.rename(dest)

            elapsed = time.time() - start
            print(f'\n  Concluido em {elapsed:.0f}s | {dest.name}')
            return str(dest)

        except Exception as e:
            print(f'\n  ERRO no download: {e}')
            print(f'  Arquivo parcial mantido em: {temp_path}')
            print(f'  Execute novamente para retomar.')
            return None

    def download_with_metadata(self, category, base_name):
        """Baixa o modelo + arquivos de metadados associados (.json, .preview.png, etc)."""
        # Buscar lista de arquivos na categoria
        r = requests.get(f'{self.server_url}/api/models/{category}', timeout=15)
        files = r.json()['files']

        # Encontrar arquivos relacionados (mesmo nome base)
        stem = Path(base_name).stem
        # Remove extensoes compostas como .preview.png
        if '.preview' in stem or '.thumb' in stem:
            stem = stem.rsplit('.', 1)[0]

        related = [f['name'] for f in files if f['name'].startswith(stem)]

        # Baixar modelo principal primeiro
        main_file = base_name
        result = self.download(category, main_file)

        # Baixar metadados (json, preview) - sem barra de progresso (pequenos)
        for fname in related:
            if fname == main_file:
                continue
            if fname.endswith(('.json', '.preview.png', '.thumb.jpg', '.yaml')):
                dest = Path(self.models_path) / category / fname
                if not dest.exists():
                    try:
                        r = requests.get(f'{self.server_url}/download/{category}/{fname}', timeout=30)
                        dest.write_bytes(r.content)
                        print(f'  + {fname}')
                    except:
                        pass

        return result

    def cached(self):
        """Lista modelos ja baixados no Colab."""
        model_exts = {'.safetensors', '.ckpt', '.pt', '.pth', '.bin'}
        cached = {}
        base = Path(self.models_path)
        for cat_dir in sorted(base.iterdir()):
            if not cat_dir.is_dir():
                continue
            files = []
            for f in sorted(cat_dir.iterdir()):
                if f.is_file() and f.suffix.lower() in model_exts:
                    files.append({
                        'name': f.name,
                        'size_mb': round(f.stat().st_size / (1024**2), 1),
                    })
            if files:
                cached[cat_dir.name] = files
        return cached

    def delete(self, category, filename):
        """Deleta um modelo do Colab para liberar espaco."""
        path = Path(self.models_path) / category / filename
        if path.exists():
            size = path.stat().st_size / (1024**2)
            path.unlink()
            # Delete associated metadata files
            stem = path.stem
            for f in path.parent.iterdir():
                if f.name.startswith(stem) and f.suffix in ('.json', '.png', '.jpg', '.yaml'):
                    f.unlink()
            print(f'Deletado: {category}/{filename} ({size:.0f} MB liberados)')
        else:
            print(f'Arquivo nao encontrado: {category}/{filename}')

    def cleanup(self, keep=None):
        """Deleta TODOS os modelos do cache exceto os listados em keep."""
        keep = keep or []
        keep_set = set(keep)
        cached = self.cached()
        freed = 0
        for cat, files in cached.items():
            for f in files:
                if f['name'] not in keep_set:
                    path = Path(self.models_path) / cat / f['name']
                    if path.exists():
                        freed += path.stat().st_size
                        path.unlink()
        print(f'Limpeza concluida: {freed/(1024**2):.0f} MB liberados')

    def disk_status(self):
        """Mostra status do disco do Colab."""
        total, used, free = shutil.disk_usage('/')
        pct = used / total * 100
        print(f'Disco Colab: {used//(1024**3)} GB usados / {total//(1024**3)} GB total ({pct:.0f}%)')
        print(f'Livre: {free//(1024**3)} GB')
        if pct > 85:
            print('AVISO: Disco quase cheio! Use mm.cleanup() para liberar espaco.')

# Instanciar o Model Manager
mm = ModelManager(LOCAL_SERVER_URL)
print('Model Manager pronto! Use as funcoes abaixo.')
print()
print('Comandos disponiveis:')
print('  mm.download("Stable-diffusion", "modelo.safetensors")  # Baixar modelo')
print('  mm.download_with_metadata("Lora", "lora.safetensors") # Baixar com metadados')
print('  mm.cached()                                           # Ver modelos no cache')
print('  mm.delete("Stable-diffusion", "modelo.safetensors")   # Deletar modelo')
print('  mm.cleanup(keep=["modelo.safetensors"])               # Limpar tudo exceto...')
print('  mm.disk_status()                                      # Ver espaco em disco')

Model Manager pronto! Use as funcoes abaixo.

Comandos disponiveis:
  mm.download("Stable-diffusion", "modelo.safetensors")  # Baixar modelo
  mm.download_with_metadata("Lora", "lora.safetensors") # Baixar com metadados
  mm.cached()                                           # Ver modelos no cache
  mm.delete("Stable-diffusion", "modelo.safetensors")   # Deletar modelo
  mm.cleanup(keep=["modelo.safetensors"])               # Limpar tudo exceto...
  mm.disk_status()                                      # Ver espaco em disco


---
## 5. Baixar Modelos

Edite os nomes dos modelos abaixo conforme sua necessidade.
Use os nomes que apareceram na listagem da celula 3.

In [ ]:
# ============================================================
# Cell 5: Baixar modelos desejados
# ============================================================
# Edite conforme necessario. Exemplos:

#@markdown ### Checkpoint (modelo base)
CHECKPOINT = 'redLilyIllu_v10.safetensors' #@param {type:"string"}

#@markdown ### LoRAs (separados por virgula, ou vazio)
LORAS = 'Breast Size Slider - Illustrious - V2_alpha1.0_rank4_noxattn_last.safetensors, Body Type Slider - Illustrious_alpha1.0_rank4_noxattn_last.safetensors, Anime_artistic_2.safetensors, Hyperrealistic_illustrious.safetensors' #@param {type:"string"}

#@markdown ### Embeddings (separados por virgula, ou vazio)
EMBEDDINGS = 'lazyneg.safetensors' #@param {type:"string"}

#@markdown ### VAE (ou vazio para usar o padrao)
VAE_MODEL = 'vae-ft-mse-840000-ema-pruned.safetensors' #@param {type:"string"}

# --- Download checkpoint ---
if CHECKPOINT.strip():
    print('=== CHECKPOINT ===')
    mm.download_with_metadata('Stable-diffusion', CHECKPOINT.strip())
    print()

# --- Download LoRAs ---
if LORAS.strip():
    print('=== LORAS ===')
    for lora in LORAS.split(','):
        lora = lora.strip()
        if lora:
            mm.download_with_metadata('Lora', lora)
    print()

# --- Download Embeddings ---
if EMBEDDINGS.strip():
    print('=== EMBEDDINGS ===')
    for emb in EMBEDDINGS.split(','):
        emb = emb.strip()
        if emb:
            mm.download_with_metadata('embeddings', emb)
    print()

# --- Download VAE ---
if VAE_MODEL.strip():
    print('=== VAE ===')
    mm.download('VAE', VAE_MODEL.strip())
    print()

# Status
print('\n=== MODELOS NO CACHE ===')
cached = mm.cached()
for cat, files in cached.items():
    for f in files:
        size = f'{f["size_mb"]:.0f} MB' if f['size_mb'] < 1024 else f'{f["size_mb"]/1024:.1f} GB'
        print(f'  [{cat}] {f["name"]}  ({size})')

mm.disk_status()

=== CHECKPOINT ===
Espaco livre: 67 GB
Baixando: Stable-diffusion/redLilyIllu_v10.safetensors
  [██████████████████████████████] 100.0% | 6617/6617 MB | 4.4 MB/s | ETA: 0s    
  Concluido em 1508s | redLilyIllu_v10.safetensors
  + redLilyIllu_v10.preview.png

=== LORAS ===
Modelo ja existe: Breast Size Slider - Illustrious - V2_alpha1.0_rank4_noxattn_last.safetensors (8 MB)
Modelo ja existe: Body Type Slider - Illustrious_alpha1.0_rank4_noxattn_last.safetensors (8 MB)
Modelo ja existe: Anime_artistic_2.safetensors (218 MB)
Modelo ja existe: Hyperrealistic_illustrious.safetensors (435 MB)

=== EMBEDDINGS ===
Modelo ja existe: lazyneg.safetensors (0 MB)

=== VAE ===
Espaco livre: 61 GB
Baixando: VAE/vae-ft-mse-840000-ema-pruned.safetensors
  [██████████████████████████████] 100.0% | 240/240 MB | 3.4 MB/s | ETA: 0s    
  Concluido em 71s | vae-ft-mse-840000-ema-pruned.safetensors


=== MODELOS NO CACHE ===
  [Lora] 69yottea_illu_v2.safetensors  (218 MB)
  [Lora] 748cmHassaku.safetensors  

---
## 5.5 Checkpoint - Salvar Ambiente no Google Drive

Rode esta celula **apos o setup completo (celulas 1-5)** para criar um snapshot do ambiente inteiro no Drive.

O que eh salvo:
- **SD.NEXT + venv** (ja estao no Drive via symlink)
- **Todos os modelos** baixados (checkpoints, LoRAs, VAE, embeddings...)
- **Dependencias pip** do Colab base (requirements_colab.txt) para restauracao rapida
- **Config do SD.NEXT** (config.json, ui-config.json)
- **Este notebook**

Nas proximas sessoes, use a celula **Quick Resume** no topo para restaurar tudo em ~1 min.

In [ ]:
# ============================================================
# CHECKPOINT - Salvar snapshot completo no Google Drive
# ============================================================
import subprocess, os, shutil, sys, json
from datetime import datetime
from pathlib import Path

DRIVE_PATH = '/content/drive/MyDrive/SD_Data'
DRIVE_SDNEXT = f'{DRIVE_PATH}/sdnext'
LOCAL_SDNEXT = '/content/sdnext'

# --- Verificar se Drive esta montado ---
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DRIVE_PATH, exist_ok=True)
saved_items = []
errors = []

# === [1/5] SALVAR DEPENDENCIAS PIP ===
print('--- [1/5] SALVANDO DEPENDENCIAS PIP ---')
try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'freeze'],
        capture_output=True, text=True, check=True
    )
    req_path = f'{DRIVE_PATH}/requirements_colab.txt'
    with open(req_path, 'w') as f:
        f.write(result.stdout)
    n_pkgs = len([l for l in result.stdout.strip().split('\n') if l.strip()])
    print(f'  {n_pkgs} pacotes salvos em requirements_colab.txt')
    saved_items.append(f'Pip: {n_pkgs} pacotes')
except Exception as e:
    errors.append(f'Pip freeze: {e}')
    print(f'  ERRO: {e}')

# === [2/5] VERIFICAR SD.NEXT NO DRIVE ===
print('\n--- [2/5] VERIFICANDO SD.NEXT NO DRIVE ---')
if os.path.islink(LOCAL_SDNEXT):
    target = os.readlink(LOCAL_SDNEXT)
    if target == DRIVE_SDNEXT:
        print(f'  Symlink OK: {LOCAL_SDNEXT} -> {DRIVE_SDNEXT}')
        saved_items.append('SD.NEXT: symlink ativo para Drive')
    else:
        print(f'  AVISO: Symlink aponta para {target}, esperado {DRIVE_SDNEXT}')
elif os.path.isdir(LOCAL_SDNEXT) and not os.path.islink(LOCAL_SDNEXT):
    print('  SD.NEXT esta local (nao no Drive). Copiando para o Drive...')
    print('  (Isso pode demorar se houver muitos arquivos)')
    if os.path.exists(DRIVE_SDNEXT):
        shutil.rmtree(DRIVE_SDNEXT)
    shutil.copytree(LOCAL_SDNEXT, DRIVE_SDNEXT, symlinks=True)
    shutil.rmtree(LOCAL_SDNEXT)
    os.symlink(DRIVE_SDNEXT, LOCAL_SDNEXT)
    print('  Copiado e symlink criado!')
    saved_items.append('SD.NEXT: copiado para Drive + symlink')
else:
    errors.append('SD.NEXT nao encontrado em /content/sdnext')
    print('  ERRO: SD.NEXT nao encontrado!')

# Verificar venv
has_venv = os.path.exists(f'{LOCAL_SDNEXT}/venv')
if has_venv:
    venv_size = sum(
        f.stat().st_size for f in Path(f'{LOCAL_SDNEXT}/venv').rglob('*') if f.is_file()
    ) / (1024**3)
    print(f'  Venv: {venv_size:.1f} GB no Drive')
    saved_items.append(f'Venv: {venv_size:.1f} GB')
else:
    print('  Venv: nao encontrado (sera criado no proximo launch)')

# === [3/5] INVENTARIO DE MODELOS ===
print('\n--- [3/5] INVENTARIO DE MODELOS ---')
model_exts = {'.safetensors', '.ckpt', '.pt', '.pth', '.bin'}
models_base = f'{LOCAL_SDNEXT}/models'
total_models = 0
total_models_size = 0
inventory = {}

if os.path.isdir(models_base):
    for cat in sorted(os.listdir(models_base)):
        cat_path = f'{models_base}/{cat}'
        if not os.path.isdir(cat_path):
            continue
        files = []
        cat_size = 0
        for fname in sorted(os.listdir(cat_path)):
            fpath = f'{cat_path}/{fname}'
            if os.path.isfile(fpath) and os.path.splitext(fname)[1].lower() in model_exts:
                fsize = os.path.getsize(fpath)
                files.append({'name': fname, 'size_mb': round(fsize / (1024**2), 1)})
                cat_size += fsize
                total_models += 1
                total_models_size += fsize
        if files:
            inventory[cat] = files
            size_str = f'{cat_size/(1024**3):.1f} GB' if cat_size > 1024**3 else f'{cat_size/(1024**2):.0f} MB'
            print(f'  [{cat}] {len(files)} modelo(s) ({size_str})')
            for f in files:
                s = f'{f["size_mb"]:.0f} MB' if f['size_mb'] < 1024 else f'{f["size_mb"]/1024:.1f} GB'
                print(f'    - {f["name"]} ({s})')

total_gb = total_models_size / (1024**3)
saved_items.append(f'Modelos: {total_models} arquivos ({total_gb:.1f} GB)')

# === [4/5] SALVAR CONFIGS ===
print('\n--- [4/5] SALVANDO CONFIGS ---')
configs_saved = 0
config_files = ['config.json', 'ui-config.json', 'params.txt', 'styles.csv']
for cfg in config_files:
    src = f'{LOCAL_SDNEXT}/{cfg}'
    if os.path.exists(src):
        dst = f'{DRIVE_PATH}/{cfg}'
        shutil.copy2(src, dst)
        print(f'  {cfg}')
        configs_saved += 1
if configs_saved > 0:
    saved_items.append(f'Configs: {configs_saved} arquivo(s)')
else:
    print('  Nenhum config encontrado (normal antes do primeiro launch)')

# Salvar notebook
print('\n--- [5/5] SALVANDO NOTEBOOK ---')
notebook_src = '/content/SD_Next_Colab.ipynb'
notebook_dst = f'{DRIVE_PATH}/SD_Next_Colab.ipynb'
if os.path.exists(notebook_src):
    shutil.copy2(notebook_src, notebook_dst)
    print(f'  Notebook salvo em Drive/SD_Data/')
    saved_items.append('Notebook: salvo')
else:
    from glob import glob as g
    notebooks = g('/content/*.ipynb')
    if notebooks:
        shutil.copy2(notebooks[0], f'{DRIVE_PATH}/{os.path.basename(notebooks[0])}')
        print(f'  {os.path.basename(notebooks[0])} salvo')
        saved_items.append('Notebook: salvo')
    else:
        print('  Notebook nao encontrado. Salve via File > Save a copy in Drive')

# === MANIFESTO DO CHECKPOINT ===
manifest = {
    'timestamp': datetime.now().isoformat(),
    'saved_items': saved_items,
    'models_inventory': inventory,
    'total_models': total_models,
    'total_models_size_gb': round(total_gb, 2),
    'has_venv': has_venv,
    'errors': errors,
}
manifest_path = f'{DRIVE_PATH}/checkpoint_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# === RESUMO FINAL ===
total_drive, used_drive, free_drive = shutil.disk_usage('/content/drive')

print('\n' + '='*60)
print('  CHECKPOINT COMPLETO!')
print('='*60)
for item in saved_items:
    print(f'  + {item}')
if errors:
    print()
    for err in errors:
        print(f'  ! ERRO: {err}')
print(f'\n  Drive: {used_drive//(1024**3)} GB usados | {free_drive//(1024**3)} GB livres')
print(f'  Local: {DRIVE_PATH}/')
print()
print('  Na proxima sessao, rode a celula "Quick Resume" no topo')
print('  e pule direto para a Celula 6 (Start SD.NEXT)')
print('='*60)

--- [1/5] SALVANDO DEPENDENCIAS PIP ---
  719 pacotes salvos em requirements_colab.txt

--- [2/5] VERIFICANDO SD.NEXT NO DRIVE ---
  Symlink OK: /content/sdnext -> /content/drive/MyDrive/SD_Data/sdnext
  Venv: nao encontrado (sera criado no proximo launch)

--- [3/5] INVENTARIO DE MODELOS ---
  [Lora] 30 modelo(s) (5.6 GB)
    - 69yottea_illu_v2.safetensors (218 MB)
    - 748cmHassaku.safetensors (244 MB)
    - 748cmSDXL.safetensors (243 MB)
    - 748cm_c_illu.safetensors (218 MB)
    - 96YOTTEA-WAI.safetensors (218 MB)
    - 96yottea-000123.safetensors (268 MB)
    - Anime_artistic_2.safetensors (218 MB)
    - Anime_in_real.safetensors (218 MB)
    - BSS_OIL_PAINTING.safetensors (218 MB)
    - Body Type Slider - Illustrious_alpha1.0_rank4_noxattn_last.safetensors (8 MB)
    - Breast Size Slider - Illustrious - V2_alpha1.0_rank4_noxattn_last.safetensors (8 MB)
    - CreateConcept_Illustrious.safetensors (218 MB)
    - DetailedEyes_V3.safetensors (89 MB)
    - Dramatic Lighting Slider.s

---
## 6. Iniciar SD.NEXT + Tunnel WebUI

Esta celula inicia o cloudflared tunnel para a WebUI e depois lanca o SD.NEXT.
A URL da WebUI aparecera nos logs abaixo.

In [ ]:
# ============================================================
# Cell 6: Iniciar cloudflared WebUI tunnel + SD.NEXT
# ============================================================
import subprocess, threading, time, re

SDNEXT_DIR = '/content/sdnext'
WEBUI_PORT = 7860

# --- Start cloudflared tunnel for WebUI ---
print('Iniciando tunnel para WebUI...')
tunnel_log = '/content/webui_tunnel.log'

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{WEBUI_PORT}'],
    stdout=open(tunnel_log, 'w'),
    stderr=subprocess.STDOUT
)

# Esperar e capturar URL do tunnel
webui_url = None
for _ in range(30):  # max 30 seconds
    time.sleep(1)
    try:
        with open(tunnel_log) as f:
            log = f.read()
        urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', log)
        if urls:
            webui_url = urls[0]
            break
    except:
        pass

if webui_url:
    print()
    print('=' * 60)
    print(f'  WebUI URL: {webui_url}')
    print('=' * 60)
    print()
    print('Abra este link no seu navegador quando o SD.NEXT terminar de carregar.')
    print('(Aguarde o SD.NEXT mostrar "Startup complete" nos logs abaixo)')
    print()
else:
    print('AVISO: Nao foi possivel capturar URL do tunnel.')
    print(f'Verifique o log: {tunnel_log}')

# --- Launch SD.NEXT ---
print('Iniciando SD.NEXT...')
print('(Primeira execucao instala dependencias - pode demorar 5-10 min)')
print()

!cd {SDNEXT_DIR} && python launch.py --listen --port {WEBUI_PORT} --use-cuda

Iniciando tunnel para WebUI...

  WebUI URL: https://public-demo-mediterranean-grade.trycloudflare.com

Abra este link no seu navegador quando o SD.NEXT terminar de carregar.
(Aguarde o SD.NEXT mostrar "Startup complete" nos logs abaixo)

Iniciando SD.NEXT...
(Primeira execucao instala dependencias - pode demorar 5-10 min)

14:25:07-647147 INFO     Starting SD.Next                                       
14:25:07-658907 INFO     Logger:                                                
                         file="/content/drive/MyDrive/SD_Data/sdnext/sdnext.log"
                         level=DEBUG host="fde9080379a1" size=79 mode=create    
14:25:07-660745 INFO     Python: version=3.12.12 platform=Linux                 
                         bin="/usr/bin/python3" venv="/usr"                     
14:25:08-039155 INFO     Version: app=sd.next updated=2026-02-07 commit=c9e21a5 
                         branch=master                                          
                         u

---
## 7. Utilitarios

In [ ]:
# ============================================================
# Cell 7: Status do disco e gerenciamento de cache
# ============================================================

print('=== STATUS DO DISCO ===')
mm.disk_status()

print('\n=== MODELOS NO CACHE ===')
cached = mm.cached()
if not cached:
    print('  Nenhum modelo no cache.')
else:
    total_cache = 0
    for cat, files in cached.items():
        for f in files:
            size = f'{f["size_mb"]:.0f} MB' if f['size_mb'] < 1024 else f'{f["size_mb"]/1024:.1f} GB'
            print(f'  [{cat}] {f["name"]}  ({size})')
            total_cache += f['size_mb']
    print(f'\n  Total em cache: {total_cache:.0f} MB ({total_cache/1024:.1f} GB)')

In [ ]:
# ============================================================
# Cell 8: Limpar modelos nao utilizados
# ============================================================
# Descomente e edite conforme necessario:

# --- Deletar um modelo especifico ---
# mm.delete('Stable-diffusion', 'nome_do_modelo.safetensors')

# --- Limpar tudo exceto modelos listados ---
# mm.cleanup(keep=['modelo_que_quero_manter.safetensors'])

# --- Limpar TUDO ---
# mm.cleanup()

print('Edite esta celula para limpar modelos do cache.')
print('Descomente a linha que deseja executar.')

In [ ]:
# ============================================================
# Cell 9: Baixar modelos adicionais durante a sessao
# ============================================================
# Use esta celula para baixar mais modelos sem parar o SD.NEXT.
# Apos o download, recarregue os modelos na WebUI (Settings > Reload).

#@markdown ### Categoria
CATEGORY = 'Lora' #@param ['Stable-diffusion', 'Lora', 'VAE', 'embeddings', 'controlnet', 'hypernetworks', 'ESRGAN', 'upscale_models', 'text_encoders', 'UNET']

#@markdown ### Nome do arquivo
FILENAME = '' #@param {type:"string"}

if FILENAME.strip():
    mm.download_with_metadata(CATEGORY, FILENAME.strip())
    mm.disk_status()
else:
    # Listar modelos da categoria
    print(f'Modelos disponiveis em [{CATEGORY}]:')
    try:
        r = requests.get(f'{LOCAL_SERVER_URL}/api/models/{CATEGORY}', timeout=15)
        data = r.json()
        for f in data['files']:
            if f['name'].endswith(('.safetensors', '.ckpt', '.pt', '.pth', '.bin')):
                size = f'{f["size_mb"]:.0f} MB' if f['size_mb'] < 1024 else f'{f["size_mb"]/1024:.1f} GB'
                print(f'  {f["name"]}  ({size})')
    except Exception as e:
        print(f'Erro: {e}')

In [ ]:
# ============================================================
# Cell 10: Salvar imagens geradas de volta para sua maquina
# ============================================================
import glob, requests, os

SDNEXT_DIR = '/content/sdnext'
OUTPUT_DIRS = [
    f'{SDNEXT_DIR}/outputs/txt2img',
    f'{SDNEXT_DIR}/outputs/img2img',
    f'{SDNEXT_DIR}/outputs/extras',
]

uploaded = 0
for out_dir in OUTPUT_DIRS:
    if not os.path.isdir(out_dir):
        continue
    for root, dirs, files in os.walk(out_dir):
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                continue
            filepath = os.path.join(root, fname)
            rel = os.path.relpath(filepath, f'{SDNEXT_DIR}/outputs')
            rel = rel.replace('\\', '/')

            try:
                with open(filepath, 'rb') as f:
                    data = f.read()
                r = requests.post(
                    f'{LOCAL_SERVER_URL}/upload/colab_{rel}',
                    data=data,
                    headers={'Content-Type': 'application/octet-stream'},
                    timeout=60
                )
                if r.status_code == 200:
                    uploaded += 1
                    print(f'  Enviado: {rel}')
                else:
                    print(f'  Erro ao enviar {rel}: {r.status_code}')
            except Exception as e:
                print(f'  Erro: {rel}: {e}')

if uploaded > 0:
    print(f'\n{uploaded} imagens enviadas para sua maquina local.')
else:
    print('Nenhuma imagem encontrada para enviar.')
    print(f'Diretorio de output: {SDNEXT_DIR}/outputs/')

---
## 8. Salvar Notebook no Google Drive

Salva este notebook no seu Google Drive para que na proxima sessao voce possa abri-lo direto de la, sem precisar fazer upload novamente.

In [ ]:
# ============================================================
# Cell 11: Salvar notebook e configs no Google Drive
# ============================================================
import shutil, os
from google.colab import drive

# Montar Drive se ainda nao estiver montado
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/SD_Data'
os.makedirs(DRIVE_PATH, exist_ok=True)

# --- Salvar este notebook no Drive ---
# O Colab salva o notebook ativo em /content/
notebook_src = '/content/SD_Next_Colab.ipynb'
notebook_dst = f'{DRIVE_PATH}/SD_Next_Colab.ipynb'

if os.path.exists(notebook_src):
    shutil.copy2(notebook_src, notebook_dst)
    print(f'Notebook salvo em: {notebook_dst}')
else:
    # Tentar encontrar o notebook por outro nome
    from glob import glob
    notebooks = glob('/content/*.ipynb')
    if notebooks:
        shutil.copy2(notebooks[0], f'{DRIVE_PATH}/{os.path.basename(notebooks[0])}')
        print(f'Notebook salvo: {os.path.basename(notebooks[0])}')
    else:
        print('Notebook nao encontrado em /content/. Salve manualmente via File > Save a copy in Drive.')

# --- Salvar config do SD.NEXT se existir ---
sdnext_config = '/content/sdnext/config.json'
if os.path.exists(sdnext_config):
    shutil.copy2(sdnext_config, f'{DRIVE_PATH}/sdnext_config_backup.json')
    print('Config do SD.NEXT salva.')

# --- Status do Drive ---
total, used, free = shutil.disk_usage('/content/drive')
print(f'\nGoogle Drive: {used//(1024**3)} GB usados | {free//(1024**3)} GB livres')

print('\n--- PROXIMA SESSAO ---')
print('1. Abra o Colab e va em File > Open notebook > Google Drive')
print(f'2. Navegue ate MyDrive/SD_Data/SD_Next_Colab.ipynb')
print('3. Seus modelos ja estarao la (via symlinks do Drive)!')
print('4. So precisa colar a nova URL do tunnel e executar.')